# INTRODUCTION
This is an attempt to train a pre-trained model using ViT and classify breast cancer

# Creating A Data Loader for this case

In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter

ModuleNotFoundError: No module named 'torch'

In [ ]:
class BiopsyImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for patient_folder in os.listdir(root_dir):
            for label_str in ["0", "1"]:
                label_folder = os.path.join(root_dir, patient_folder, label_str)
                if not os.path.isdir(label_folder):
                    continue
                label = int(label_str)
                for file in os.listdir(label_folder):
                    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        full_path = os.path.join(label_folder, file)
                        self.samples.append((full_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
# Chemin racine
root_dir = '../data/image50_clahe'

# Transforms d'entrée pour ViT
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Charger le dataset
dataset = BiopsyImageDataset(root_dir=root_dir, transform=transform)

# Compter les labels
labels = [label for _, label in dataset.samples]
label_counts = Counter(labels)
print(f"🔢 Classe 0 : {label_counts[0]}, Classe 1 : {label_counts[1]}")

# Créer un poids inversement proportionnel à la fréquence
class_weights = {label: 1.0 / count for label, count in label_counts.items()}
sample_weights = [class_weights[label] for _, label in dataset.samples]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# DataLoader équilibré
loader = DataLoader(
    dataset,
    batch_size=32,
    sampler=sampler
)

🔢 Classe 0 : 198738, Classe 1 : 78786


In [ ]:
import random

# Suppose que dataset = BiopsyImageDataset(...)
all_samples = dataset.samples

# Séparer selon les classes
class_0 = [sample for sample in all_samples if sample[1] == 0]
class_1 = [sample for sample in all_samples if sample[1] == 1]

# Mélanger et réduire
random.seed(42)
random.shuffle(class_0)
random.shuffle(class_1)

subset_0 = class_0[:5000]
subset_1 = class_1[:5000]

# Fusionner et mélanger
subset_samples = subset_0 + subset_1
random.shuffle(subset_samples)


In [ ]:
class LimitedBiopsyDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt")
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(label)
        }


In [ ]:
subset_dataset = LimitedBiopsyDataset(subset_samples, transform=transform)

train_loader = DataLoader(
    subset_dataset,
    batch_size=32,
    shuffle=True  # plus besoin de sampler ici
)


In [ ]:
from torch.utils.data import random_split

train_set, val_set = random_split(subset_dataset, [8000, 2000])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)


In [ ]:
import torch
from transformers import ViTForImageClassification
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from tqdm import tqdm

from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=2,
    ignore_mismatched_sizes=True
)

from transformers import ViTFeatureExtractor

# Charger le feature extractor associé au ViT
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 5

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\tanjo\projetcs\ruban_rose\ruban_rose\boobs\Lib\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


## Entrainement

In [ ]:
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    total = 0
    correct = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in loop:
        inputs = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)

        outputs = model(pixel_values=inputs)
        loss = criterion(outputs.logits, labels)

        # Optimisation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item(), acc=correct/total)

    epoch_acc = correct / total
    print(f"✅ Epoch {epoch+1} done — Acc: {epoch_acc:.4f} — Loss: {running_loss:.4f}")


Epoch 1/5:   0%|          | 1/250 [01:02<4:17:26, 62.03s/it, acc=0.469, loss=0.747]


KeyboardInterrupt: 